# Clase práctica: AutoML con PyCaret

**Objetivo:** recorrer el ciclo completo de un proyecto de *machine learning* con muy poco código: preparar datos, comparar modelos, ajustarlos, evaluarlos, combinarlos y dejarlos listos para producción.

| Parte | Problema | Dataset | Idea clave |
|---|---|---|---|
| 1 | Clasificación | `diabetes` (Pima Indians) | El flujo completo, de `setup()` a `save_model()` |
| 2 | Regresión | `insurance` | La API orientada a objetos y la transformación del objetivo |
| 3 | Clustering | `jewellery` | Aprendizaje no supervisado con la misma lógica |
| 4 | Ejercicios | — | Practicar por tu cuenta |

> **Flujo PyCaret:** `setup` → `compare_models` → `create_model` → `tune_model` → `plot_model` → `predict_model` → `finalize_model` → `save_model`

## 0. Preparación del entorno

PyCaret 3.3.2 (última versión estable) funciona con **Python 3.9, 3.10 y 3.11**. Con Python 3.12 o superior la instalación falla, así que conviene crear un entorno dedicado:

```bash
# Opción A: conda
conda create -n pycaret python=3.11 -y && conda activate pycaret
pip install pycaret==3.3.2 jupyter

# Opción B: uv
uv venv -p 3.11 .venv && source .venv/bin/activate
uv pip install pycaret==3.3.2 jupyter
```

Si ya estás en un entorno con Python 3.11, basta con ejecutar la celda siguiente (quita el `#`).

In [ ]:
# !pip install -q pycaret==3.3.2

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import pycaret
import sklearn

print("Python   :", sys.version.split()[0])
print("PyCaret  :", pycaret.__version__)
print("sklearn  :", sklearn.__version__)
pd.set_option("display.max_columns", 50)

---
# Parte 1 · Clasificación: ¿el paciente tiene diabetes?

Usaremos el dataset **Pima Indians Diabetes** (768 pacientes, 8 variables clínicas). La variable objetivo vale `1` si el paciente desarrolló diabetes y `0` si no.

### 1.1 Cargar los datos

In [ ]:
from pycaret.datasets import get_data

df = get_data("diabetes", verbose=False)

# Los nombres originales son muy largos: los acortamos para trabajar cómodos
df.columns = ["embarazos", "glucosa", "presion", "pliegue_piel",
              "insulina", "imc", "pedigri", "edad", "diabetes"]
df.head()

### 1.2 Exploración rápida

Antes de automatizar nada, **mira los datos**. AutoML no sustituye el sentido común.

In [ ]:
print(df.shape)
df.describe().T.round(2)

In [ ]:
# ¿Está balanceada la variable objetivo?
df["diabetes"].value_counts(normalize=True).round(3)

In [ ]:
# Valores imposibles: nadie tiene glucosa, presión o IMC igual a 0.
# En este dataset, un 0 significa "dato no registrado".
cols_cero = ["glucosa", "presion", "pliegue_piel", "insulina", "imc"]
(df[cols_cero] == 0).sum()

In [ ]:
# Los marcamos como faltantes (NaN) y dejamos que PyCaret los impute en setup()
df[cols_cero] = df[cols_cero].replace(0, np.nan)
df.isna().sum()

### 1.3 Reservar datos "nuevos"

Separamos un 5 % de filas que **PyCaret nunca verá** durante el experimento. Al final las usaremos para simular la llegada de pacientes nuevos en producción.

In [ ]:
datos_nuevos = df.sample(frac=0.05, random_state=42)
df_modelo = df.drop(datos_nuevos.index).reset_index(drop=True)
datos_nuevos = datos_nuevos.reset_index(drop=True)

print("Para modelar :", df_modelo.shape)
print("Datos nuevos :", datos_nuevos.shape)

### 1.4 `setup()`: el corazón de PyCaret

`setup()` **siempre va primero**. Con una sola llamada:

- divide los datos en *train* y *test* (estratificado),
- infiere el tipo de cada columna,
- construye un *pipeline* de preprocesamiento (imputación, codificación, escalado…),
- configura la validación cruzada.

Parámetros que usaremos:

| Parámetro | Qué hace |
|---|---|
| `target` | Columna a predecir |
| `session_id` | Semilla: hace el experimento reproducible |
| `train_size` | Proporción de entrenamiento (el resto es *test*) |
| `numeric_imputation` | Cómo rellenar faltantes numéricos (`mean`, `median`, …) |
| `normalize` | Escala las variables (útil para modelos lineales, KNN, SVM) |
| `fold` | Número de *folds* de validación cruzada |

In [ ]:
from pycaret.classification import *

s = setup(
    data=df_modelo,
    target="diabetes",
    session_id=123,
    train_size=0.8,
    numeric_imputation="median",
    normalize=True,
    fold=5,
)

`get_config()` permite inspeccionar todo lo que creó `setup()`: los conjuntos de datos, el *pipeline*, la configuración de CV…

In [ ]:
X_train_t = get_config("X_train_transformed")
print(X_train_t.shape)
X_train_t.head()

In [ ]:
# El pipeline de preprocesamiento es un objeto de scikit-learn
get_config("pipeline")

### 1.5 `compare_models()`: entrenar y comparar todo de una vez

Entrena **todos** los modelos disponibles con validación cruzada y los ordena por la métrica que elijamos. En problemas médicos como este, el **AUC** es más informativo que la *accuracy*.

Excluimos `ridge` y `svm` porque **no calculan probabilidades** (`predict_proba`): las necesitamos para el *blend* con voto suave (`method="soft"`), para ajustar el umbral de decisión y para el AUC sobre datos nuevos.

In [ ]:
# ridge y svm no tienen predict_proba: los excluimos para poder combinar modelos con voto "soft"
top3 = compare_models(sort="AUC", n_select=3, exclude=["ridge", "svm"])

In [ ]:
# pull() devuelve la última tabla de resultados como DataFrame
ranking = pull()
ranking.head()

In [ ]:
top3

### 1.6 `create_model()`: entrenar un modelo concreto

`models()` lista los identificadores disponibles. Con `create_model()` entrenamos uno con validación cruzada y vemos la métrica **en cada *fold***: esto nos dice si el modelo es estable.

In [ ]:
models()[["Name", "Turbo"]].head(16)

In [ ]:
rf = create_model("rf")

### 1.7 `tune_model()`: ajustar hiperparámetros

Hace una búsqueda aleatoria (*random search*) sobre una rejilla predefinida.

- `optimize`: métrica a maximizar.
- `n_iter`: cuántas combinaciones probar.
- `choose_better=True`: si el modelo ajustado es peor que el original, devuelve el original.

In [ ]:
rf_tuned = tune_model(rf, optimize="AUC", n_iter=20, choose_better=True)

In [ ]:
# ¿Qué hiperparámetros cambió?
rf_tuned.get_params()

### 1.8 `plot_model()`: entender el modelo

Algunos gráficos útiles: `auc`, `confusion_matrix`, `pr`, `feature`, `learning`, `threshold`, `calibration`, `class_report`.

In [ ]:
plot_model(rf_tuned, plot="auc")

In [ ]:
plot_model(rf_tuned, plot="confusion_matrix")

In [ ]:
plot_model(rf_tuned, plot="feature")

In [ ]:
plot_model(rf_tuned, plot="threshold")

> **Tip:** en Jupyter, `evaluate_model(modelo)` abre un panel interactivo con todos los gráficos. Pruébalo descomentando la celda.

In [ ]:
# evaluate_model(rf_tuned)

### 1.9 Combinar modelos

Tres estrategias de *ensemble*:

| Función | Idea |
|---|---|
| `ensemble_model(m, method="Bagging" / "Boosting")` | Muchas copias del mismo modelo |
| `blend_models([m1, m2, m3])` | Votación entre modelos distintos |
| `stack_models([m1, m2, m3])` | Un meta-modelo aprende a combinar las predicciones |

In [ ]:
blend = blend_models(top3, optimize="AUC", method="soft")

In [ ]:
stack = stack_models(top3, optimize="AUC")

### 1.10 `predict_model()`: evaluar en el conjunto de *test*

Hasta ahora todas las métricas eran de validación cruzada sobre *train*. Sin argumento `data`, `predict_model()` evalúa sobre el **20 % de test** que separó `setup()`. Si la métrica en test se parece a la de CV, el modelo **generaliza**.

In [ ]:
candidatos = {"RF ajustado": rf_tuned, "Blend": blend, "Stack": stack}

resultados = []
for nombre, modelo in candidatos.items():
    predict_model(modelo, verbose=False)
    fila = pull().iloc[0]
    fila["Modelo"] = nombre
    resultados.append(fila)

tabla_test = pd.DataFrame(resultados).set_index("Modelo")[["Accuracy", "AUC", "Recall", "Prec.", "F1"]]
tabla_test

> **¿Más complejo = mejor?** No necesariamente. Combinar modelos suele ayudar, pero no siempre gana a un buen modelo individual bien ajustado. Por eso **se mide, no se supone**.
>
> Compara esta tabla con la de validación cruzada: en CV el blend y el stack quedan ligeramente por delante, pero en *test* gana el RF ajustado. Las diferencias son menores que la variación entre *folds* (±0.04), así que en la práctica son empates.
>
> Aquí elegimos el de mejor AUC en *test*. Ojo: si comparas muchísimos candidatos sobre el mismo *test*, acabas sobreajustando también el *test*. Lo ideal es decidir con la validación cruzada y usar el *test* como confirmación final.

In [ ]:
mejor_nombre = tabla_test["AUC"].idxmax()
modelo_elegido = candidatos[mejor_nombre]
print("Modelo elegido:", mejor_nombre)

### 1.11 Ajustar el umbral de decisión

Por defecto se predice `1` si la probabilidad es ≥ 0.5. En un cribado médico quizá prefiramos **no dejar pasar casos positivos** (más *recall*), aunque aumenten los falsos positivos. Comparemos varios umbrales:

In [ ]:
for umbral in [0.5, 0.4, 0.3]:
    predict_model(modelo_elegido, probability_threshold=umbral, verbose=False)
    m = pull().iloc[0]
    print(f"Umbral {umbral:.2f} -> Recall {m['Recall']:.3f} | Precisión {m['Prec.']:.3f} | F1 {m['F1']:.3f}")

### 1.12 `finalize_model()` y predicción sobre datos nuevos

`finalize_model()` **reentrena el modelo con todos los datos** (train + test). Úsalo solo al final, cuando ya hayas elegido el modelo: después ya no hay test para evaluarlo.

In [ ]:
modelo_final = finalize_model(modelo_elegido)
modelo_final

In [ ]:
# Los datos nuevos llegan sin la etiqueta: la quitamos para simular producción
X_nuevos = datos_nuevos.drop(columns="diabetes")

pred = predict_model(modelo_final, data=X_nuevos)
pred[["glucosa", "imc", "edad", "prediction_label", "prediction_score"]].head(10)

In [ ]:
# Como en este ejercicio sí conocemos la verdad, podemos comprobar el acierto
from sklearn.metrics import accuracy_score, roc_auc_score

pred_prob = predict_model(modelo_final, data=X_nuevos, raw_score=True, verbose=False)
print("Accuracy:", round(accuracy_score(datos_nuevos["diabetes"], pred_prob["prediction_label"]), 3))
print("AUC     :", round(roc_auc_score(datos_nuevos["diabetes"], pred_prob["prediction_score_1"]), 3))

> Con solo unas 38 filas, estas métricas son **muy ruidosas**: un par de aciertos o fallos las mueven varios puntos. No las compares a la ligera con las de *test* (146 filas) ni con las de CV.

### 1.13 Guardar y cargar el modelo

`save_model()` guarda **el pipeline completo** (preprocesamiento + modelo) en un `.pkl`. Quien lo cargue puede pasarle datos crudos, sin repetir la imputación ni el escalado.

In [ ]:
save_model(modelo_final, "modelo_diabetes")

In [ ]:
modelo_cargado = load_model("modelo_diabetes")

paciente = pd.DataFrame([{
    "embarazos": 2, "glucosa": 165, "presion": 74, "pliegue_piel": np.nan,
    "insulina": np.nan, "imc": 36.1, "pedigri": 0.45, "edad": 41,
}])
predict_model(modelo_cargado, data=paciente)[["prediction_label", "prediction_score"]]

---
# Parte 2 · Regresión: ¿cuánto costará el seguro médico?

Dataset **insurance**: 1 338 asegurados con edad, sexo, IMC, hijos, si fuma y región. Queremos predecir `charges` (coste anual).

Esta vez usaremos la **API orientada a objetos**: en lugar de funciones globales, creamos un objeto `RegressionExperiment` y llamamos a sus métodos. Los verbos son exactamente los mismos, pero:

- puedes tener **varios experimentos a la vez** sin que se pisen,
- el código es más explícito y fácil de llevar a scripts o producción.

In [ ]:
from pycaret.datasets import get_data
from pycaret.regression import RegressionExperiment

seguros = get_data("insurance", verbose=False)
print(seguros.shape)
seguros.head()

In [ ]:
# La variable objetivo tiene una cola larga a la derecha
seguros["charges"].describe().round(0)

In [ ]:
seguros.groupby("smoker")["charges"].mean().round(0)

### 2.1 `setup()` con transformación del objetivo

`transform_target=True` transforma `charges` para que su distribución sea más simétrica. Las predicciones se devuelven automáticamente **en la escala original**.

- `transform_target_method="quantile"`: mapea el objetivo a una distribución normal usando sus cuantiles.
- El método por defecto (`"yeo-johnson"`) puede fallar con un `BracketError` de scipy cuando el objetivo toma valores tan grandes como estos (decenas de miles). Por eso usamos `quantile`.

Las variables categóricas (`sex`, `smoker`, `region`) se detectan y codifican solas.

In [ ]:
exp_reg = RegressionExperiment()
exp_reg.setup(
    data=seguros,
    target="charges",
    session_id=123,
    normalize=True,
    transform_target=True,
    transform_target_method="quantile",
    fold=5,
)

In [ ]:
exp_reg.get_config("X_train_transformed").head()

### 2.2 Comparar, ajustar y evaluar
En regresión las métricas habituales son **MAE**, **RMSE** y **R²**. Ordenamos por MAE (error medio en dólares, fácil de explicar).

In [ ]:
mejores_reg = exp_reg.compare_models(sort="MAE", n_select=3)

In [ ]:
mejor_reg = exp_reg.tune_model(mejores_reg[0], optimize="MAE", n_iter=20, choose_better=True)

In [ ]:
exp_reg.plot_model(mejor_reg, plot="residuals")

In [ ]:
exp_reg.plot_model(mejor_reg, plot="error")

In [ ]:
exp_reg.plot_model(mejor_reg, plot="feature")

### 2.3 Predicción sobre *test* y sobre un cliente nuevo

In [ ]:
pred_test = exp_reg.predict_model(mejor_reg)
pred_test[["age", "bmi", "smoker", "charges", "prediction_label"]].head()

In [ ]:
final_reg = exp_reg.finalize_model(mejor_reg)

clientes = pd.DataFrame([
    {"age": 35, "sex": "female", "bmi": 24.0, "children": 1, "smoker": "no",  "region": "northwest"},
    {"age": 35, "sex": "female", "bmi": 24.0, "children": 1, "smoker": "yes", "region": "northwest"},
])
exp_reg.predict_model(final_reg, data=clientes)

> Misma persona, solo cambia `smoker`: la diferencia en el coste predicho muestra lo que el modelo ha aprendido.

---
# Parte 3 · Clustering: segmentar clientes

Dataset **jewellery**: 505 clientes con `Age`, `Income`, `SpendingScore` y `Savings`. No hay variable objetivo: queremos descubrir **grupos** de clientes parecidos.

La lógica es la misma: `setup()` (sin `target`) → `create_model()` → `plot_model()` → `assign_model()`.

In [ ]:
from pycaret.clustering import ClusteringExperiment

joyeria = get_data("jewellery", verbose=False)
joyeria.describe().round(2)

In [ ]:
exp_clu = ClusteringExperiment()
exp_clu.setup(data=joyeria, normalize=True, session_id=123)

In [ ]:
# Método del codo: ¿cuántos clusters tienen sentido?
kmeans_tmp = exp_clu.create_model("kmeans")
exp_clu.plot_model(kmeans_tmp, plot="elbow")

In [ ]:
kmeans = exp_clu.create_model("kmeans", num_clusters=4)

In [ ]:
exp_clu.plot_model(kmeans, plot="distribution")

In [ ]:
# assign_model añade la etiqueta de cluster a cada cliente
segmentos = exp_clu.assign_model(kmeans)
resumen = segmentos.groupby("Cluster").mean(numeric_only=True).astype(float).round(2)
resumen["n_clientes"] = segmentos["Cluster"].value_counts()
resumen

> **Para discutir:** ¿qué nombre le pondrías a cada segmento? (p. ej. "jóvenes que gastan mucho", "ahorradores de mayor edad"…)

---
# Parte 4 · Ejercicios

Resuélvelos en las celdas vacías. Las pistas están al final de cada enunciado.

**Ejercicio 1 · Desbalanceo.** Repite el `setup()` de diabetes añadiendo `fix_imbalance=True` (aplica SMOTE). Compara el *Recall* del mejor modelo con el que obtuviste antes.
*Pista: crea un `ClassificationExperiment()` nuevo para no pisar el experimento de la Parte 1.*

In [ ]:
from pycaret.classification import ClassificationExperiment

# Mismo setup que en la Parte 1 (misma semilla y train_size => mismo split de test) + SMOTE
exp_smote = ClassificationExperiment()
exp_smote.setup(
    data=df_modelo,
    target="diabetes",
    session_id=123,
    train_size=0.8,
    numeric_imputation="median",
    normalize=True,
    fold=5,
    fix_imbalance=True,
    verbose=False,
)
mejor_smote = exp_smote.compare_models(sort="AUC", exclude=["ridge", "svm"])

# Recall en test: modelo de la Parte 1 vs. mejor modelo con SMOTE
predict_model(modelo_elegido, verbose=False)
recall_antes = pull().iloc[0]["Recall"]
exp_smote.predict_model(mejor_smote, verbose=False)
recall_smote = exp_smote.pull().iloc[0]["Recall"]

print(f"Recall sin SMOTE ({mejor_nombre}): {recall_antes:.3f}")
print(f"Recall con SMOTE ({type(mejor_smote).__name__}): {recall_smote:.3f}")

**Ejercicio 2 · Comparar solo algunos modelos.** En el experimento de regresión, compara únicamente `lr`, `ridge`, `rf` y `lightgbm`.
*Pista: parámetro `include=[...]` de `compare_models()`.*

In [ ]:
exp_reg.compare_models(include=["lr", "ridge", "rf", "lightgbm"], sort="MAE")

**Ejercicio 3 · Métrica propia.** Añade al experimento de regresión una métrica "% de predicciones con error menor a 2 000 $" y vuelve a comparar.
*Pista: `exp_reg.add_metric(id, name, score_func, greater_is_better=True)`; `score_func` recibe `(y_true, y_pred, **kwargs)`.*

In [ ]:
# **kwargs es obligatorio: PyCaret le pasa argumentos extra (p. ej. sample_weight).
# Sin él, la métrica falla en silencio y aparece como 0.0 en la tabla.
def pct_error_menor_2000(y_true, y_pred, **kwargs):
    """Proporción de predicciones cuyo error absoluto es menor a 2 000 $."""
    errores = np.abs(np.asarray(y_true) - np.asarray(y_pred))  # asarray evita que pandas alinee por índice
    return np.mean(errores < 2000)

exp_reg.add_metric(
    id="pct_err_2000",
    name="% err < 2000",
    score_func=pct_error_menor_2000,
    greater_is_better=True,
)

# La nueva métrica aparece como una columna más y podemos ordenar por ella
exp_reg.compare_models(sort="% err < 2000")

**Ejercicio 4 · Ingeniería de variables.** En diabetes, prueba `setup(..., remove_multicollinearity=True, feature_selection=True, n_features_to_select=5)`. ¿Empeora mucho el AUC usando solo 5 variables? ¿Cuáles se quedan?
*Pista: `get_config("X_train_transformed").columns`.*

In [ ]:
exp_fs = ClassificationExperiment()
exp_fs.setup(
    data=df_modelo,
    target="diabetes",
    session_id=123,
    train_size=0.8,
    numeric_imputation="median",
    normalize=True,
    fold=5,
    remove_multicollinearity=True,
    feature_selection=True,
    n_features_to_select=5,
    verbose=False,
)
print("Variables que se quedan:", list(exp_fs.get_config("X_train_transformed").columns))

mejor_fs = exp_fs.compare_models(sort="AUC", exclude=["ridge", "svm"])
auc_fs = exp_fs.pull().iloc[0]["AUC"]
print(f"AUC CV con 8 variables: {ranking['AUC'].iloc[0]:.4f}")
print(f"AUC CV con 5 variables: {auc_fs:.4f}")

**Ejercicio 5 · Reto.** Elige otro dataset con `get_data("index")` (por ejemplo `credit`, `bank` o `employee`), identifica el tipo de problema y recorre el flujo completo hasta `save_model()`.

In [ ]:
# Dataset "credit": ¿el cliente dejará de pagar su tarjeta? (default = 1) -> clasificación binaria
credito = get_data("credit", verbose=False)
print(credito.shape)
print(credito["default"].value_counts(normalize=True).round(3))

exp_cred = ClassificationExperiment()
exp_cred.setup(data=credito, target="default", session_id=123, normalize=True, fold=5, verbose=False)

mejor_cred = exp_cred.compare_models(sort="AUC", exclude=["ridge", "svm"])
mejor_cred = exp_cred.tune_model(mejor_cred, optimize="AUC", n_iter=10, choose_better=True)
exp_cred.plot_model(mejor_cred, plot="auc")
exp_cred.predict_model(mejor_cred)  # métricas en test

final_cred = exp_cred.finalize_model(mejor_cred)
exp_cred.save_model(final_cred, "modelo_credit")

---
# Resumen: chuleta de PyCaret

| Paso | Función | Para qué |
|---|---|---|
| 1 | `setup(data, target, session_id)` | Preprocesar y configurar el experimento |
| 2 | `compare_models(sort, n_select)` | Ranking de todos los modelos |
| 3 | `create_model("id")` | Entrenar un modelo concreto con CV |
| 4 | `tune_model(m, optimize, n_iter)` | Ajustar hiperparámetros |
| 5 | `blend_models` / `stack_models` / `ensemble_model` | Combinar modelos |
| 6 | `plot_model(m, plot)` / `evaluate_model(m)` | Diagnosticar |
| 7 | `predict_model(m, data)` | Evaluar en test o predecir |
| 8 | `finalize_model(m)` | Reentrenar con todos los datos |
| 9 | `save_model` / `load_model` | Persistir el pipeline completo |
| — | `pull()` / `get_config()` / `models()` | Recuperar tablas, objetos y catálogo |

**Buenas prácticas**

1. Mira los datos antes de automatizar (los ceros "imposibles" de diabetes).
2. Fija siempre `session_id` para poder reproducir resultados.
3. Elige la métrica según el problema, no por costumbre.
4. Compara CV con test: si difieren mucho, hay sobreajuste.
5. `finalize_model()` solo al final.

**Documentación:** https://pycaret.gitbook.io/docs